# 🔬 PySCF: Zero to Hero
## A Comprehensive Guide to Quantum Chemistry in Python

---

**Welcome!** This notebook is your complete guide to PySCF (Python-based Simulations of Chemistry Framework) — from first principles to expert-level quantum chemistry. By the end, you'll be able to:
- Run Hartree-Fock (HF) and DFT calculations
- Perform post-HF correlated methods (MP2, CCSD, CCSD(T))
- Compute molecular properties: dipole moments, polarizabilities, NMR
- Run geometry optimizations and frequency calculations
- Perform excited state calculations (TDDFT, EOM-CCSD)
- Work with multireference methods (CASSCF, CASPT2, NEVPT2)
- Use periodic boundary conditions (solid-state / band structure)
- Apply embedding and QM/MM methods

**Prerequisites:** Basic Python, some quantum chemistry background helpful

**Installation:**
```bash
pip install pyscf
# GPU support:
pip install gpu4pyscf
# Extra modules:
pip install pyscf[geomopt]   # geometry optimization
pip install pyscf[cosmo]     # solvation
```

---

## 📚 Table of Contents

1. [Chapter 1: Core Concepts & Architecture](#chapter1)
2. [Chapter 2: Molecules & Basis Sets](#chapter2)
3. [Chapter 3: Hartree-Fock (HF)](#chapter3)
4. [Chapter 4: Density Functional Theory (DFT)](#chapter4)
5. [Chapter 5: Molecular Properties](#chapter5)
6. [Chapter 6: Post-HF Correlation Methods](#chapter6)
7. [Chapter 7: Coupled Cluster Theory](#chapter7)
8. [Chapter 8: Geometry Optimization & Frequencies](#chapter8)
9. [Chapter 9: Excited States — TDDFT & EOM](#chapter9)
10. [Chapter 10: Multireference Methods (CASSCF/CASPT2/NEVPT2)](#chapter10)
11. [Chapter 11: Relativistic Methods](#chapter11)
12. [Chapter 12: Periodic Systems (PBC/Solid State)](#chapter12)
13. [Chapter 13: Solvation & Embedding](#chapter13)
14. [Chapter 14: Wavefunction Analysis](#chapter14)
15. [Chapter 15: Advanced Topics & Workflows](#chapter15)

---
<a id='chapter1'></a>
# Chapter 1: Core Concepts & Architecture 🏗️

## What is PySCF?

PySCF is a quantum chemistry package written in Python/C. It is:
- **Modular**: each method is a separate Python class
- **Flexible**: easy to mix, extend, and prototype new methods
- **Fast**: hot loops in C/Fortran, optional GPU acceleration
- **Open source**: MIT license, highly active development

## The Central Abstraction

```
mol = gto.Mole()          ← Molecule: atoms, basis, charge, spin
    ↓
mf = scf.RHF(mol)         ← Method object (RHF, UKS, RCCSD, CASSCF...)
    ↓
mf.kernel()               ← Run the calculation → returns energy
    ↓
mf.mo_coeff               ← Results: MO coefficients
mf.mo_energy              ← MO energies
mf.make_rdm1()            ← 1-particle density matrix
```

## Key Modules

| Module | Purpose |
|--------|---------|
| `pyscf.gto` | Molecule definition, basis set integrals |
| `pyscf.scf` | HF: RHF, UHF, ROHF, GHF |
| `pyscf.dft` | DFT: RKS, UKS, ROKS |
| `pyscf.mp` | MP2, MP3 |
| `pyscf.cc` | CCSD, CCSD(T), EOM-CCSD |
| `pyscf.ci` | FCI, CISD |
| `pyscf.mcscf` | CASSCF, CASCI |
| `pyscf.mrpt` | CASPT2, NEVPT2 |
| `pyscf.tdscf` | TDDFT, TDHF |
| `pyscf.prop` | Properties: NMR, EPR, polarizability |
| `pyscf.solvent` | PCM, COSMO solvation |
| `pyscf.pbc` | Periodic boundary conditions |
| `pyscf.qmmm` | QM/MM embedding |

In [ ]:
# Verify installation
import pyscf
print(f"PySCF version: {pyscf.__version__}")

# Core imports
from pyscf import gto, scf, dft, mp, cc, ci, mcscf, mrpt, tdscf
from pyscf import lo, tools, lib
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Optional: set number of threads
lib.num_threads(4)  # Use 4 CPU threads

print("All imports successful! ✅")
print(f"NumPy: {np.__version__}")

In [ ]:
# ============================================================
# PySCF Logging & Output Control
# ============================================================
from pyscf import lib

# Verbosity levels
# 0 = silent, 1 = minimal, 3 = normal, 5 = debug, 9 = very verbose

# Set globally
lib.logger.QUIET = False

# Set per-object
mol = gto.Mole()
mol.verbose = 3   # normal output

# Redirect output to file
mol.output = '/tmp/pyscf.log'

# Or suppress output
mol.verbose = 0

print("Logging configured")
print("Tip: mol.verbose = 4 gives detailed SCF convergence info")

---
<a id='chapter2'></a>
# Chapter 2: Molecules & Basis Sets ⚛️

Everything starts with defining your molecule and choosing a basis set.

In [ ]:
# ============================================================
# Building a Molecule — gto.Mole
# ============================================================
from pyscf import gto

# Method 1: Inline string (most readable)
mol = gto.Mole()
mol.atom = '''
    O   0.000000   0.000000   0.117176
    H   0.000000   0.757440  -0.468706
    H   0.000000  -0.757440  -0.468706
'''
mol.basis = 'cc-pVDZ'
mol.charge = 0
mol.spin   = 0         # 2S (0 = singlet, 1 = doublet, 2 = triplet)
mol.verbose = 3
mol.build()

print(f"Water molecule:")
print(f"  Atoms:    {mol.natm}")
print(f"  Electrons:{mol.nelectron}")
print(f"  Basis:    {mol.basis}")
print(f"  nAO:      {mol.nao_nr()} (contracted)")
print(f"  Symmetry: {mol.topgroup}")

In [ ]:
# ============================================================
# Multiple Atom Input Formats
# ============================================================

# Format 1: List of tuples (symbol, (x, y, z))
mol1 = gto.M(
    atom = [('C', (0, 0, 0)),
            ('O', (0, 0, 1.21))],
    basis = '6-31G*',
    verbose = 0
)
print(f"CO (list format): {mol1.natm} atoms, {mol1.nao_nr()} AOs")

# Format 2: Z-matrix (internal coordinates)
mol2 = gto.M(
    atom = '''
        C
        O 1 1.21
    ''',
    basis = '6-31G*',
    verbose = 0
)
print(f"CO (z-matrix):    {mol2.natm} atoms, {mol2.nao_nr()} AOs")

# Format 3: From XYZ string
xyz_string = """
6
ethylene
C   0.000  0.000  0.000
C   0.000  0.000  1.339
H   0.000  0.928 -0.554
H   0.000 -0.928 -0.554
H   0.000  0.928  1.893
H   0.000 -0.928  1.893
"""
mol3 = gto.M(
    atom = gto.fromstring(xyz_string, toAngstrom=True),
    basis = '6-31G',
    verbose = 0
)
print(f"Ethylene (xyz):   {mol3.natm} atoms, {mol3.nao_nr()} AOs")

# From XYZ file
# mol4 = gto.M(atom='molecule.xyz', basis='6-31G*')

In [ ]:
# ============================================================
# Basis Sets — Overview
# ============================================================
print("Basis Set Hierarchy (accuracy ↑, cost ↑):")
print()

basis_guide = [
    ("STO-3G",       "Minimal",          "Very fast, qualitative only"),
    ("3-21G",        "Split-valence",     "Fast, qualitative"),
    ("6-31G",        "Split-valence",     "Popular for organic"),
    ("6-31G*",       "Pople + d pol.",    "Standard for optimization"),
    ("6-31G**",      "Pople + d,p pol.", "+ H polarization"),
    ("6-311G**",     "Triple-zeta Pople", "Better for properties"),
    ("cc-pVDZ",      "Dunning DZ",        "Correlated methods, small"),
    ("cc-pVTZ",      "Dunning TZ",        "Standard for correlated"),
    ("cc-pVQZ",      "Dunning QZ",        "High accuracy, expensive"),
    ("aug-cc-pVDZ",  "+ diffuse",         "Anions, excited states"),
    ("aug-cc-pVTZ",  "+ diffuse",         "Accurate properties"),
    ("def2-SVP",     "Ahlrichs DZ",       "All elements incl. metals"),
    ("def2-TZVP",    "Ahlrichs TZ",       "Standard for TM/metals"),
    ("def2-TZVPP",   "Ahlrichs TZ+pol",  "High-quality metal calcs"),
    ("def2-QZVP",    "Ahlrichs QZ",       "Near-CBS for metals"),
    ("cc-pCVTZ",     "Core-valence",      "Core correlation"),
    ("ANO-RCC",      "ANO relativistic", "Heavy elements + SO"),
]

print(f"{'Name':15s} {'Type':18s} {'Use'}")
print("-" * 65)
for name, btype, use in basis_guide:
    print(f"  {name:13s} {btype:18s} {use}")

In [ ]:
# ============================================================
# Mixed Basis Sets & Custom Basis
# ============================================================

# Different basis on different atoms
mol_mixed = gto.M(
    atom = 'Fe 0 0 0; O 0 0 2.0; O 0 0 -2.0',
    basis = {
        'Fe': 'def2-TZVP',   # Triple-zeta for metal center
        'O':  'cc-pVDZ',     # Double-zeta for ligands
    },
    charge = 0,
    spin   = 4,              # FeO2 high-spin: 5 unpaired d electrons
    verbose = 0
)
print(f"Mixed basis (Fe/O): {mol_mixed.nao_nr()} AOs")

# Custom basis from BSE (Basis Set Exchange) format
# PySCF can parse Molpro, Gaussian, and NWChem formats
custom_basis = gto.basis.parse('''
#  STO-3G for H
H     S
      3.4252509    0.1543290
      0.6239137    0.5353281
      0.1688554    0.4446345
''')
mol_custom = gto.M(
    atom = 'H 0 0 0; H 0 0 0.74',
    basis = {'H': custom_basis},
    verbose = 0
)
print(f"Custom basis H2: {mol_custom.nao_nr()} AOs")

In [ ]:
# ============================================================
# Inspecting Integrals
# ============================================================
mol = gto.M(atom='H 0 0 0; H 0 0 0.74', basis='sto-3g', verbose=0)

# 1-electron integrals
S  = mol.intor('int1e_ovlp')    # Overlap matrix
T  = mol.intor('int1e_kin')     # Kinetic energy
V  = mol.intor('int1e_nuc')     # Nuclear attraction
H_core = T + V                  # Core Hamiltonian

print(f"H2 (STO-3G):")
print(f"  Basis functions: {mol.nao_nr()}")
print(f"  Overlap matrix S:\n{S.round(4)}")
print(f"  Core Hamiltonian H:\n{H_core.round(4)}")

# 2-electron integrals (electron repulsion)
# Shape: (nao, nao, nao, nao) — can be huge!
eri = mol.intor('int2e')
print(f"\n  ERI shape: {eri.shape}")
print(f"  ERI size: {eri.nbytes / 1024:.1f} KB")

# Nuclear repulsion
E_nuc = mol.energy_nuc()
print(f"  Nuclear repulsion: {E_nuc:.6f} Hartree")

In [ ]:
# ============================================================
# Molecular Symmetry
# ============================================================
# PySCF can exploit symmetry to speed up calculations

mol_sym = gto.M(
    atom = '''
        N  0.000  0.000  0.549
        N  0.000  0.000 -0.549
    ''',
    basis = 'cc-pVTZ',
    symmetry = True,     # Detect and use symmetry
    verbose = 0
)
print(f"N2 symmetry group: {mol_sym.topgroup}")
print(f"Point group:       {mol_sym.groupname}")
print(f"Irreducible representations: {mol_sym.irrep_name}")

# Water
mol_h2o_sym = gto.M(
    atom = 'O 0 0 0.117; H 0 0.757 -0.469; H 0 -0.757 -0.469',
    basis = 'cc-pVDZ',
    symmetry = True,
    verbose = 0
)
print(f"\nH2O symmetry group: {mol_h2o_sym.topgroup}")
print(f"Point group:        {mol_h2o_sym.groupname}")

---
<a id='chapter3'></a>
# Chapter 3: Hartree-Fock (HF) Theory 🌊

HF is the starting point for all ab initio methods. It approximates the wavefunction as a single Slater determinant and optimizes the molecular orbitals self-consistently.

In [ ]:
# ============================================================
# RHF — Restricted Hartree-Fock (closed-shell)
# ============================================================
from pyscf import gto, scf

mol = gto.M(
    atom = 'O 0 0 0.117; H 0 0.757 -0.469; H 0 -0.757 -0.469',
    basis = 'cc-pVDZ',
    verbose = 4
)

mf = scf.RHF(mol)
mf.conv_tol    = 1e-10    # Energy convergence threshold (Hartree)
mf.conv_tol_grad = 1e-6   # Gradient convergence threshold
mf.max_cycle   = 100      # Max SCF iterations

E_hf = mf.kernel()

print(f"\n{'='*50}")
print(f"RHF energy:         {E_hf:.10f} Hartree")
print(f"Converged:          {mf.converged}")
print(f"HOMO index:         {mol.nelectron//2 - 1}")
print(f"HOMO energy:        {mf.mo_energy[mol.nelectron//2 - 1]:.6f} Hartree")
print(f"LUMO energy:        {mf.mo_energy[mol.nelectron//2]:.6f} Hartree")
print(f"HOMO-LUMO gap:      {(mf.mo_energy[mol.nelectron//2] - mf.mo_energy[mol.nelectron//2-1])*27.211:.4f} eV")

In [ ]:
# ============================================================
# Inspecting MO Results
# ============================================================
import numpy as np

n_occ = mol.nelectron // 2
n_vir = mf.mo_energy.shape[0] - n_occ

print(f"Molecular Orbitals:")
print(f"  Occupied: {n_occ}")
print(f"  Virtual:  {n_vir}")
print()
print(f"{'MO':>5} {'Type':>8} {'Energy (Eh)':>14} {'Energy (eV)':>13} {'Occ.':>6}")
print("-" * 55)
for i, (e, occ) in enumerate(zip(mf.mo_energy, mf.mo_occ)):
    mo_type = 'occ' if occ > 0 else 'vir'
    marker = ' ← HOMO' if i == n_occ-1 else (' ← LUMO' if i == n_occ else '')
    print(f"  {i:3d} {mo_type:>8} {e:14.6f} {e*27.211:13.4f} {occ:6.1f}{marker}")

# MO Coefficients matrix: columns = MOs, rows = AOs
print(f"\nMO coefficient matrix shape: {mf.mo_coeff.shape}  (nAO x nMO)")

In [ ]:
# ============================================================
# Mulliken & Löwdin Population Analysis
# ============================================================
print("Population Analysis:")
print()

# Mulliken charges
mulliken = mf.mulliken_pop()
print("Mulliken Analysis:")
print(f"  (printed above in kernel output)")

# Get charges explicitly
pop, chg = mf.mulliken_pop(verbose=0)
print("\nMulliken Charges:")
for i, atom in enumerate(mol._atom):
    print(f"  {atom[0]:3s}: {chg[i]:+.4f}")

# Löwdin charges (more robust)
pop_lowdin, chg_lowdin = mf.mulliken_pop(s=mol.intor('int1e_ovlp'), verbose=0)
print("\nNote: For Löwdin, use mf.mulliken_meta() or scf.hf.mulliken_pop_cond()")

# Dipole moment
dip = mf.dip_moment()
print(f"\nDipole moment: {dip} (Debye)")
print(f"  |μ| = {np.linalg.norm(dip):.4f} D")

In [ ]:
# ============================================================
# UHF — Unrestricted HF (open-shell)
# ============================================================
# Use for radicals, triplets, any system with unpaired electrons

# Methyl radical (doublet, 1 unpaired electron)
mol_rad = gto.M(
    atom = '''
        C  0.000  0.000  0.000
        H  0.000  1.079  0.000
        H  0.934 -0.540  0.000
        H -0.934 -0.540  0.000
    ''',
    basis = 'cc-pVDZ',
    charge = 0,
    spin = 1,     # 2S = 1 → doublet
    verbose = 3
)

mf_u = scf.UHF(mol_rad)
E_uhf = mf_u.kernel()

print(f"\nUHF energy: {E_uhf:.8f} Hartree")
print(f"<S²>: {mf_u.spin_square()[0]:.4f}  (expected: {mf_u.spin_square()[1]:.4f})")
# <S²> tells you spin contamination — should be close to S(S+1)

# Alpha and beta MO energies are different in UHF
print(f"Alpha HOMO: {mf_u.mo_energy[0][mol_rad.nelec[0]-1]:.6f} Eh")
print(f"Beta  HOMO: {mf_u.mo_energy[1][mol_rad.nelec[1]-1]:.6f} Eh")

In [ ]:
# ============================================================
# SCF Convergence Tricks
# ============================================================

mol = gto.M(atom='Fe 0 0 0', basis='def2-SVP', spin=4, charge=0, verbose=0)
mf = scf.UHF(mol)

# 1. DIIS (Direct Inversion of Iterative Subspace) — default
mf.diis = True
mf.diis_space = 8   # DIIS subspace size

# 2. Level shifting — adds energy gap to avoid orbital switching
mf.level_shift = 0.2    # eV shift applied to virtual orbitals

# 3. Damping — mix old and new density matrix
mf.damp = 0.5          # 0=no damping, 1=full old density

# 4. Custom initial guess
# dm0 = mf.get_init_guess(mol, key='atom')  # atomic density
# dm0 = mf.get_init_guess(mol, key='huckel') # extended Hückel
# mf.kernel(dm0=dm0)

# 5. Broken symmetry initial guess for antiferromagnetic coupling
# mf.kernel(dm0=None)  # default

E = mf.kernel()
print(f"Fe atom UHF energy: {E:.6f} Eh")
print(f"<S²>: {mf.spin_square()[0]:.4f}")

---
<a id='chapter4'></a>
# Chapter 4: Density Functional Theory (DFT) 🎯

DFT replaces the expensive wavefunction with the electron density as the fundamental variable. The exchange-correlation functional is the key approximation.

In [ ]:
# ============================================================
# RKS — Restricted Kohn-Sham DFT
# ============================================================
from pyscf import dft

mol = gto.M(
    atom = 'O 0 0 0.117; H 0 0.757 -0.469; H 0 -0.757 -0.469',
    basis = 'def2-TZVP',
    verbose = 3
)

# B3LYP — the most widely used hybrid functional
mf_b3lyp = dft.RKS(mol)
mf_b3lyp.xc = 'b3lyp'
mf_b3lyp.grids.level = 3     # Integration grid: 1 (coarse) to 9 (fine)
E_b3lyp = mf_b3lyp.kernel()

print(f"\nB3LYP/def2-TZVP energy: {E_b3lyp:.10f} Eh")

In [ ]:
# ============================================================
# DFT Functional Zoo
# ============================================================
print("Jacob's Ladder of DFT Functionals (accuracy ↑, cost ↑):")
print()

functionals = [
    ("Rung 1: LDA",          ["lda, vwn",        "svwn",       "lda,pw"]),
    ("Rung 2: GGA",          ["pbe",              "blyp",       "bp86",       "pbe,pbe"]),
    ("Rung 3: meta-GGA",     ["tpss",             "m06l",       "scan",       "r2scan"]),
    ("Rung 4: Hybrid",       ["b3lyp",            "pbe0",       "m062x",      "wb97x-d"]),
    ("Rung 4: Range-sep.",   ["cam-b3lyp",        "wb97",       "hse06",      "lc-whpbe"]),
    ("Rung 5: Double-hybrid",["b2plyp",           "b2gpplyp",   "dsd-blyp"]),
]

for level, funcs in functionals:
    print(f"  {level}")
    print(f"    Examples: {', '.join(funcs)}")
    print()

print("Dispersion corrections (add '-d3' or '-d3bj'):")
print("  b3lyp-d3, pbe-d3bj, m06-2x-d3, r2scan-d3bj")

In [ ]:
# ============================================================
# Comparing Functionals on Water
# ============================================================
mol = gto.M(
    atom = 'O 0 0 0.117; H 0 0.757 -0.469; H 0 -0.757 -0.469',
    basis = 'cc-pVTZ',
    verbose = 0
)

functionals_to_test = [
    ('HF',           None),
    ('LDA (SVWN)',   'lda,vwn'),
    ('PBE',          'pbe'),
    ('BLYP',         'blyp'),
    ('B3LYP',        'b3lyp'),
    ('PBE0',         'pbe0'),
    ('CAM-B3LYP',    'cam-b3lyp'),
    ('M06-2X',       'm062x'),
    ('wB97X-D',      'wb97x-d'),
]

results = []
for name, xc in functionals_to_test:
    if xc is None:
        mf = scf.RHF(mol)
    else:
        mf = dft.RKS(mol)
        mf.xc = xc
        mf.grids.level = 4
    mf.verbose = 0
    E = mf.kernel()
    dip = mf.dip_moment(verbose=0)
    results.append({'Method': name, 'Energy (Eh)': round(E, 8), 'Dipole (D)': round(np.linalg.norm(dip), 4)})

df = pd.DataFrame(results).set_index('Method')
print(df.to_string())
print(f"\nExperimental dipole of water: 1.8546 D")

In [ ]:
# ============================================================
# Dispersion Corrections (DFT-D3)
# ============================================================
# Long-range dispersion is missing from most functionals
# D3 correction adds an empirical van der Waals term

mol_dimer = gto.M(
    atom = '''
        C   0.0  0.0  0.0
        C   0.0  0.0  1.4
        H   0.0  1.02 -0.57
        H   0.0 -1.02 -0.57
        H   0.0  1.02  1.97
        H   0.0 -1.02  1.97
    ''',
    basis = 'cc-pVDZ',
    verbose = 0
)

# Without dispersion
mf_plain = dft.RKS(mol_dimer)
mf_plain.xc = 'b3lyp'
mf_plain.verbose = 0
E_plain = mf_plain.kernel()

# With D3BJ dispersion
mf_d3 = dft.RKS(mol_dimer)
mf_d3.xc = 'b3lyp-d3bj'  # or 'b3lyp' + d3 via dftd3 package
mf_d3.verbose = 0
E_d3 = mf_d3.kernel()

print(f"B3LYP:       {E_plain:.8f} Eh")
print(f"B3LYP-D3BJ:  {E_d3:.8f} Eh")
print(f"Dispersion contribution: {(E_d3 - E_plain)*2625.5:.4f} kJ/mol")

In [ ]:
# ============================================================
# Density-Fitting (RI/DF) for Speed
# ============================================================
# RI (Resolution of Identity) approximates 4-center 2e integrals
# as products of 3-center integrals → 10-100x speedup for large systems

from pyscf import df

mol = gto.M(
    atom = 'O 0 0 0.117; H 0 0.757 -0.469; H 0 -0.757 -0.469',
    basis = 'def2-TZVP',
    verbose = 0
)

# Standard RKS
import time
mf1 = dft.RKS(mol)
mf1.xc = 'pbe0'
mf1.verbose = 0
t0 = time.time()
E1 = mf1.kernel()
t1 = time.time() - t0

# DF-RKS (density fitting)
mf2 = dft.RKS(mol).density_fit()
mf2.xc = 'pbe0'
mf2.with_df.auxbasis = 'def2-TZVP-jkfit'  # auxiliary basis
mf2.verbose = 0
t0 = time.time()
E2 = mf2.kernel()
t2 = time.time() - t0

print(f"PBE0/def2-TZVP:    {E1:.8f} Eh  ({t1:.3f}s)")
print(f"DF-PBE0/def2-TZVP: {E2:.8f} Eh  ({t2:.3f}s)")
print(f"Energy difference: {abs(E1-E2)*2625.5:.6f} kJ/mol (should be tiny)")

---
<a id='chapter5'></a>
# Chapter 5: Molecular Properties 📐

Once you have converged MOs, you can compute many molecular properties.

In [ ]:
# ============================================================
# Dipole Moment, Quadrupole, and Higher Multipoles
# ============================================================
from pyscf import gto, scf, dft
from pyscf.prop import dip_moment, polarizability

mol = gto.M(
    atom = 'O 0 0 0.117; H 0 0.757 -0.469; H 0 -0.757 -0.469',
    basis = 'aug-cc-pVTZ',   # augmented basis important for properties
    verbose = 0
)
mf = dft.RKS(mol)
mf.xc = 'b3lyp'
mf.verbose = 0
mf.kernel()

# Dipole moment
dip = mf.dip_moment(mol, mf.make_rdm1(), unit='Debye', verbose=0)
print(f"Dipole moment: {dip}")
print(f"  |μ| = {np.linalg.norm(dip):.4f} D  (exp: 1.855 D)")

# Quadrupole moment
from pyscf.prop.multipole import Multipole
mult = Multipole(mf)
quad = mult.kernel()
print(f"\nQuadrupole moment tensor (a.u.):")
print(quad.round(4))

In [ ]:
# ============================================================
# Polarizability
# ============================================================
# α = -d²E/dF²  (response of dipole to external electric field)

from pyscf.prop import polarizability as polar_module

# Static polarizability
polar = polar_module.Polarizability(mf)
alpha = polar.kernel()
print(f"Static polarizability tensor (a.u.):")
print(alpha.round(4))
alpha_iso = np.trace(alpha) / 3
print(f"Isotropic polarizability: {alpha_iso:.4f} a.u.  ({alpha_iso*0.14818:.4f} Å³)")
print(f"(Experimental: ~1.45 Å³ for water)")

In [ ]:
# ============================================================
# NMR Shielding Constants
# ============================================================
from pyscf.prop import nmr

# Water NMR
mol_nmr = gto.M(
    atom = 'O 0 0 0.117; H 0 0.757 -0.469; H 0 -0.757 -0.469',
    basis = 'pcSseg-2',   # Purpose-built NMR basis
    verbose = 0
)
mf_nmr = scf.RHF(mol_nmr)
mf_nmr.verbose = 0
mf_nmr.kernel()

# NMR shielding
nmr_calc = nmr.RHF(mf_nmr)
shielding = nmr_calc.kernel()

print("NMR Isotropic Shielding Constants (ppm):")
for i, atom in enumerate(mol_nmr._atom):
    sigma_iso = np.trace(shielding[i]) / 3
    print(f"  {atom[0]}{i+1}: σ = {sigma_iso:.2f} ppm")
print("\nNote: Chemical shift = σ_ref - σ_molecule")
print("  For ¹H NMR: use TMS (tetramethylsilane) as reference")

In [ ]:
# ============================================================
# ESP (Electrostatic Potential) Charges
# ============================================================
from pyscf.prop import esp_charges

# Fit point charges to reproduce molecular ESP
# Essential for force field parameterization (RESP charges)

mol = gto.M(
    atom = 'O 0 0 0.117; H 0 0.757 -0.469; H 0 -0.757 -0.469',
    basis = '6-31G*',
    verbose = 0
)
mf = scf.RHF(mol)
mf.verbose = 0
mf.kernel()

# Generate Merz-Kollman grid and fit ESP charges
from pyscf.prop.espchrg import esp_fit
q = esp_fit(mol, mf.make_rdm1())
print("ESP/Merz-Kollman Charges:")
for i, atom in enumerate(mol._atom):
    print(f"  {atom[0]}{i+1}: q = {q[i]:+.4f} e")
print(f"  Sum: {sum(q):.6f} (should be {mol.charge})")

In [ ]:
# ============================================================
# Visualizing Electron Density and MOs
# ============================================================
from pyscf.tools import cubegen

mol = gto.M(
    atom = 'O 0 0 0.117; H 0 0.757 -0.469; H 0 -0.757 -0.469',
    basis = 'cc-pVDZ',
    verbose = 0
)
mf = scf.RHF(mol)
mf.verbose = 0
mf.kernel()

# Write electron density to cube file (for visualization in VMD/PyMol)
cubegen.density(mol, '/tmp/water_density.cube', mf.make_rdm1())
print("Electron density written to /tmp/water_density.cube")

# Write HOMO to cube file
homo_idx = mol.nelectron // 2 - 1
cubegen.orbital(mol, '/tmp/water_homo.cube', mf.mo_coeff[:, homo_idx])
print(f"HOMO (MO #{homo_idx}) written to /tmp/water_homo.cube")

# Write LUMO
lumo_idx = mol.nelectron // 2
cubegen.orbital(mol, '/tmp/water_lumo.cube', mf.mo_coeff[:, lumo_idx])
print(f"LUMO (MO #{lumo_idx}) written to /tmp/water_lumo.cube")

print("\nVisualize with: vmd water_density.cube")
print("Or: py3Dmol (pip install py3Dmol)")

---
<a id='chapter6'></a>
# Chapter 6: Post-HF Correlation Methods 🧮

HF misses electron correlation. Post-HF methods improve this at increasing computational cost.

In [ ]:
# ============================================================
# MP2 — Møller-Plesset 2nd Order Perturbation Theory
# ============================================================
# Cost: O(N⁵)  — good balance of cost and accuracy
from pyscf import mp

mol = gto.M(
    atom = 'O 0 0 0.117; H 0 0.757 -0.469; H 0 -0.757 -0.469',
    basis = 'cc-pVTZ',
    verbose = 0
)
mf = scf.RHF(mol)
mf.verbose = 0
E_hf = mf.kernel()

# MP2
mp2 = mp.MP2(mf)
mp2.verbose = 3
E_corr, t2 = mp2.kernel()
E_mp2 = E_hf + E_corr

print(f"\nHF energy:          {E_hf:.10f} Eh")
print(f"MP2 correlation:    {E_corr:.10f} Eh")
print(f"MP2 total:          {E_mp2:.10f} Eh")
print(f"\nOpposite-spin (OS): {mp2.e_corr_ss:.8f} Eh  (less well-described)")  
print(f"Same-spin (SS):     {mp2.e_corr_os:.8f} Eh  (better described)")

In [ ]:
# ============================================================
# DF-MP2 — Density Fitting MP2 (much faster)
# ============================================================
# For large systems, DF-MP2 is 10-100x faster than conventional MP2

mp2_df = mp.DFMP2(mf)
mp2_df.verbose = 0
E_corr_df, _ = mp2_df.kernel()
print(f"DF-MP2 correlation: {E_corr_df:.10f} Eh")
print(f"Difference from MP2: {abs(E_corr - E_corr_df)*2625.5:.6f} kJ/mol")

In [ ]:
# ============================================================
# Full CI (FCI) — Exact within the basis set
# ============================================================
# FCI scales exponentially — only feasible for small systems
# but is the exact reference for benchmarking
from pyscf import fci

mol_small = gto.M(
    atom = 'H 0 0 0; H 0 0 0.74',
    basis = 'sto-3g',
    verbose = 0
)
mf_h2 = scf.RHF(mol_small)
mf_h2.verbose = 0
E_hf_h2 = mf_h2.kernel()

# FCI
cisolver = fci.FCI(mf_h2)
cisolver.verbose = 3
E_fci, civec = cisolver.kernel()

print(f"\nH₂/STO-3G:")
print(f"  HF energy:  {E_hf_h2:.8f} Eh")
print(f"  FCI energy: {E_fci:.8f} Eh")
print(f"  Correlation:{(E_fci-E_hf_h2)*2625.5:.4f} kJ/mol")
print(f"\nFCI civec shape: {civec.shape}")
print(f"CI coefficients: {civec.flatten()[:4].round(6)}")  # dominant configs

In [ ]:
# ============================================================
# CISD — Configuration Interaction Singles and Doubles
# ============================================================
from pyscf import ci

mol = gto.M(
    atom = 'O 0 0 0.117; H 0 0.757 -0.469; H 0 -0.757 -0.469',
    basis = 'cc-pVDZ',
    verbose = 0
)
mf = scf.RHF(mol)
mf.verbose = 0
E_hf = mf.kernel()

cisd = ci.CISD(mf)
cisd.verbose = 3
E_corr_cisd, civec_cisd = cisd.kernel()
E_cisd = E_hf + E_corr_cisd

print(f"\nCISD correlation energy: {E_corr_cisd:.8f} Eh")
print(f"CISD total energy:       {E_cisd:.8f} Eh")

# T1 diagnostic (measure of multi-reference character)
# T1 > 0.02 suggests significant correlation / MR character
t1_diag = np.linalg.norm(civec_cisd[1]) / np.sqrt(mol.nelectron)
print(f"\nT1 diagnostic (approx): {t1_diag:.4f}")
print(f"  (Rule of thumb: > 0.02 → multireference effects)")

---
<a id='chapter7'></a>
# Chapter 7: Coupled Cluster Theory 🎯

Coupled cluster is the gold standard of quantum chemistry. CCSD(T) with a large basis set gives "chemical accuracy" (~1 kcal/mol).

In [ ]:
# ============================================================
# CCSD — Coupled Cluster Singles and Doubles
# ============================================================
# Cost: O(N⁶) — manageable for ~20-50 atoms with modern hardware
from pyscf import cc

mol = gto.M(
    atom = 'O 0 0 0.117; H 0 0.757 -0.469; H 0 -0.757 -0.469',
    basis = 'cc-pVTZ',
    verbose = 0
)
mf = scf.RHF(mol)
mf.verbose = 0
E_hf = mf.kernel()

mycc = cc.CCSD(mf)
mycc.verbose = 4
mycc.conv_tol = 1e-9
E_corr_ccsd, t1, t2 = mycc.kernel()
E_ccsd = E_hf + E_corr_ccsd

print(f"\nCCSD results:")
print(f"  Correlation energy:  {E_corr_ccsd:.10f} Eh")
print(f"  Total energy:        {E_ccsd:.10f} Eh")
print(f"  T1 norm:             {np.linalg.norm(t1):.6f}")
print(f"  T1 diagnostic:       {np.linalg.norm(t1)/np.sqrt(2*mol.nelectron//2):.4f}")

In [ ]:
# ============================================================
# CCSD(T) — The Gold Standard
# ============================================================
# Perturbative triples correction: O(N⁷)

E_ccsdt = mycc.ccsd_t()
E_ccsd_t_total = E_ccsd + E_ccsdt

print(f"CCSD(T):")
print(f"  Triples correction:  {E_ccsdt:.10f} Eh")
print(f"  CCSD(T) total:       {E_ccsd_t_total:.10f} Eh")
print(f"\nEnergy ladder (kcal/mol relative to HF):")
for name, E in [('HF', E_hf), ('CCSD', E_ccsd), ('CCSD(T)', E_ccsd_t_total)]:
    delta = (E - E_hf) * 627.509
    print(f"  {name:12s}: {E:.8f} Eh  ({delta:+.4f} kcal/mol from HF)")

In [ ]:
# ============================================================
# CBS Extrapolation — Complete Basis Set Limit
# ============================================================
# Correlation energy converges slowly with basis set size
# CBS extrapolation uses 2 calculations to estimate the limit

def cbs_extrapolate_helgaker(E_small, E_large, n_small, n_large):
    """
    Helgaker CBS extrapolation for correlation energy.
    E(n) = E_CBS + A * n^(-3)
    
    n_small, n_large: cardinal numbers (DZ=2, TZ=3, QZ=4)
    """
    E_cbs = (n_large**3 * E_large - n_small**3 * E_small) / (n_large**3 - n_small**3)
    return E_cbs

mol = gto.M(
    atom = 'O 0 0 0.117; H 0 0.757 -0.469; H 0 -0.757 -0.469',
    basis = 'cc-pVDZ',
    verbose = 0
)

# MP2/cc-pVDZ
mf2 = scf.RHF(mol); mf2.verbose = 0; mf2.kernel()
mp2_dz = mp.MP2(mf2); mp2_dz.verbose = 0; e_corr_dz, _ = mp2_dz.kernel()

# MP2/cc-pVTZ
mol.basis = 'cc-pVTZ'; mol.build()
mf3 = scf.RHF(mol); mf3.verbose = 0; mf3.kernel()
mp2_tz = mp.MP2(mf3); mp2_tz.verbose = 0; e_corr_tz, _ = mp2_tz.kernel()

# CBS extrapolation
e_corr_cbs = cbs_extrapolate_helgaker(e_corr_dz, e_corr_tz, 2, 3)

print("CBS Extrapolation for MP2/H₂O:")
print(f"  MP2/cc-pVDZ: {e_corr_dz:.8f} Eh")
print(f"  MP2/cc-pVTZ: {e_corr_tz:.8f} Eh")
print(f"  CBS limit:   {e_corr_cbs:.8f} Eh")

In [ ]:
# ============================================================
# CCSD Density Matrix & Natural Orbitals
# ============================================================
mol = gto.M(
    atom = 'O 0 0 0.117; H 0 0.757 -0.469; H 0 -0.757 -0.469',
    basis = 'cc-pVDZ',
    verbose = 0
)
mf = scf.RHF(mol); mf.verbose = 0; mf.kernel()
mycc = cc.CCSD(mf); mycc.verbose = 0; mycc.kernel()

# CCSD density matrix
dm1 = mycc.make_rdm1()
print(f"CCSD 1-RDM shape: {dm1.shape}")

# Natural orbitals (diagonalize 1-RDM)
occ, natorb = np.linalg.eigh(dm1)
occ = occ[::-1]  # sort descending
natorb = natorb[:, ::-1]

print("\nNatural Orbital Occupation Numbers:")
for i, n in enumerate(occ):
    bar = '█' * int(n * 10)
    label = 'occ' if n > 1.99 else ('active' if n > 0.01 else 'vir')
    print(f"  NO {i:2d}: {n:.6f} [{label}] {bar}")

---
<a id='chapter8'></a>
# Chapter 8: Geometry Optimization & Vibrational Frequencies 🔬

In [ ]:
# ============================================================
# Geometry Optimization
# ============================================================
# Requires: pip install geometric  (or berny)

from pyscf import gto, dft
from pyscf.geomopt.berny_solver import optimize  # or geometric_solver

# Starting geometry (slightly distorted water)
mol = gto.M(
    atom = '''
        O   0.000   0.000   0.100
        H   0.000   0.800  -0.500
        H   0.000  -0.800  -0.500
    ''',
    basis = 'def2-SVP',
    verbose = 3
)

mf = dft.RKS(mol)
mf.xc = 'b3lyp'
mf.grids.level = 3

try:
    mol_eq = optimize(mf, maxsteps=100)
    print("\nOptimized geometry (Angstrom):")
    for i, (sym, coords) in enumerate(mol_eq._atom):
        c = np.array(coords) * 0.529177  # Bohr → Angstrom
        print(f"  {sym}: {c}")

    # Bond lengths
    pos = np.array([c for _, c in mol_eq._atom]) * 0.529177
    r_OH1 = np.linalg.norm(pos[1] - pos[0])
    r_OH2 = np.linalg.norm(pos[2] - pos[0])
    angle = np.degrees(np.arccos(
        np.dot(pos[1]-pos[0], pos[2]-pos[0]) /
        (r_OH1 * r_OH2)
    ))
    print(f"\n  r(O-H1) = {r_OH1:.4f} Å  (exp: 0.9572 Å)")
    print(f"  r(O-H2) = {r_OH2:.4f} Å")
    print(f"  ∠HOH    = {angle:.2f}°  (exp: 104.52°)")

except ImportError:
    print("Install geometric or berny: pip install geometric berny")
    print("Demonstrating gradient calculation instead...")
    mf.kernel()
    g = mf.nuc_grad_method().kernel()
    print(f"Gradient:\n{g}")

In [ ]:
# ============================================================
# Analytic Gradients
# ============================================================
# Gradients are needed for optimization and dynamics

mol = gto.M(
    atom = 'O 0 0 0.117; H 0 0.757 -0.469; H 0 -0.757 -0.469',
    basis = 'cc-pVDZ',
    verbose = 0
)
mf = dft.RKS(mol)
mf.xc = 'b3lyp'
mf.verbose = 0
mf.kernel()

# Analytical gradient
grad = mf.nuc_grad_method()
g = grad.kernel()

print("Analytical Gradient (Hartree/Bohr):")
for i, (sym, _) in enumerate(mol._atom):
    print(f"  {sym}{i+1}: {g[i].round(6)}")

g_norm = np.sqrt(np.sum(g**2))
print(f"\nGradient RMS: {np.sqrt(np.mean(g**2)):.2e}")
print(f"Gradient max: {np.max(np.abs(g)):.2e}")
print(f"(Converged geometry has both < 1e-4 to 1e-5)")

In [ ]:
# ============================================================
# Vibrational Frequencies (Hessian)
# ============================================================
from pyscf.hessian import rks as rks_hess

mol = gto.M(
    atom = 'O 0 0 0.117; H 0 0.757 -0.469; H 0 -0.757 -0.469',
    basis = 'cc-pVDZ',
    verbose = 0
)
mf = dft.RKS(mol)
mf.xc = 'b3lyp'
mf.grids.level = 4
mf.verbose = 0
mf.kernel()

# Compute Hessian
hessian = mf.Hessian()
hessian.verbose = 3
H = hessian.kernel()

# Normal modes from Hessian
from pyscf.hessian.thermo import harmonic_analysis, thermo

freq_info = harmonic_analysis(mol, H)
print("Vibrational Frequencies (cm⁻¹):")
for i, freq in enumerate(freq_info['freq_wavenumber']):
    label = 'imaginary' if freq < 0 else ''
    print(f"  Mode {i+1}: {freq:.2f} cm⁻¹ {label}")
print(f"\nExperimental H₂O: 1595, 3657, 3756 cm⁻¹")
print(f"(B3LYP harmonic frequencies should be scaled by ~0.96)")

In [ ]:
# ============================================================
# Thermochemistry (G, H, S, Cv)
# ============================================================
# Standard thermochemical analysis from frequencies

from pyscf.hessian.thermo import thermo

thermo_data = thermo(mf, freq_info['freq_au'], 298.15, 101325)

print("Thermochemistry (B3LYP/cc-pVDZ, 298.15 K):")
keys_of_interest = [
    'ZPE',       # Zero-point energy
    'E_0K',      # E + ZPE
    'H',         # Enthalpy
    'G',         # Gibbs free energy
    'S',         # Entropy
    'Cv',        # Heat capacity at constant V
]
for key in keys_of_interest:
    if key in thermo_data:
        val = thermo_data[key]
        print(f"  {key:8s}: {val[0]:.8f} {val[1]}")

---
<a id='chapter9'></a>
# Chapter 9: Excited States — TDDFT & EOM-CCSD 🌈

In [ ]:
# ============================================================
# TDDFT — Time-Dependent DFT
# ============================================================
# Most popular method for excitation energies of large molecules
from pyscf import tdscf

# Formaldehyde (H2CO) — has well-known n→π* and π→π* transitions
mol = gto.M(
    atom = '''
        C   0.000  0.000  0.000
        O   0.000  0.000  1.220
        H   0.000  0.945 -0.540
        H   0.000 -0.945 -0.540
    ''',
    basis = 'aug-cc-pVDZ',   # augmented basis for excited states!
    verbose = 0
)

mf = dft.RKS(mol)
mf.xc = 'cam-b3lyp'         # Range-separated hybrid: better for CT states
mf.grids.level = 4
mf.verbose = 0
mf.kernel()

# TDDFT
td = mf.TDDFT()             # or tdscf.TDDFT(mf)
td.nstates = 10             # Number of excited states
td.verbose = 3
td.kernel()

print(f"\n{'State':>6} {'Energy (eV)':>12} {'Osc. Str.':>11} {'Assignment'}")
print("-" * 55)
for i, (E, osc) in enumerate(zip(td.e, td.oscillator_strength())):
    E_eV = E * 27.211
    E_nm = 1239.8 / E_eV
    print(f"  S{i+1:2d}   {E_eV:10.4f}  {osc:11.6f}  ({E_nm:.1f} nm)")

In [ ]:
# ============================================================
# Plotting UV-Vis Absorption Spectrum
# ============================================================
def plot_uvvis(excitation_energies_eV, oscillator_strengths,
               sigma=0.2, n_points=1000, x_range=(1, 10)):
    """Simulate UV-Vis spectrum using Gaussian broadening."""
    E_grid = np.linspace(*x_range, n_points)
    spectrum = np.zeros_like(E_grid)
    
    for E, f in zip(excitation_energies_eV, oscillator_strengths):
        spectrum += f * np.exp(-(E_grid - E)**2 / (2 * sigma**2))
    
    return E_grid, spectrum

E_eV = np.array(td.e) * 27.211
osc = td.oscillator_strength()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Stick spectrum
ax1.vlines(E_eV, 0, osc, colors='b', lw=2)
ax1.set_xlabel('Excitation Energy (eV)')
ax1.set_ylabel('Oscillator Strength')
ax1.set_title('TDDFT Stick Spectrum — Formaldehyde')
ax1.grid(True, alpha=0.3)

# Broadened spectrum (nm)
E_nm = 1239.8 / E_eV
E_grid_eV, spectrum = plot_uvvis(E_eV, osc, sigma=0.2, x_range=(3, 12))
E_grid_nm = 1239.8 / E_grid_eV
idx_sort = np.argsort(E_grid_nm)
ax2.plot(E_grid_nm[idx_sort], spectrum[idx_sort], 'b-', lw=2)
ax2.axvline(x=304, color='r', linestyle='--', alpha=0.5, label='n→π* (exp ~304 nm)')
ax2.set_xlabel('Wavelength (nm)')
ax2.set_ylabel('Intensity (arb. u.)')
ax2.set_title('Simulated UV-Vis Spectrum')
ax2.legend()
ax2.set_xlim(100, 500)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# EOM-CCSD — Equation-of-Motion CCSD
# ============================================================
# More accurate than TDDFT, especially for charge-transfer and doubly-excited states
# Cost: O(N⁶) per state — limited to small molecules

mol_small = gto.M(
    atom = '''
        C   0.000  0.000  0.000
        O   0.000  0.000  1.220
        H   0.000  0.945 -0.540
        H   0.000 -0.945 -0.540
    ''',
    basis = 'cc-pVDZ',
    verbose = 0
)
mf_cc = scf.RHF(mol_small)
mf_cc.verbose = 0
mf_cc.kernel()

mycc = cc.CCSD(mf_cc)
mycc.verbose = 0
mycc.kernel()

# EOM-EE-CCSD: Excitation energies
myeom = mycc.EOMEESingles()
myeom.nroots = 5  # first 5 singlet excited states
myeom.verbose = 3
E_exc, vecs = myeom.kernel()

print("\nEOM-EE-CCSD Excitation Energies:")
for i, E in enumerate(E_exc):
    print(f"  State {i+1}: {E*27.211:.4f} eV  ({1239.8/(E*27.211):.1f} nm)")

In [ ]:
# ============================================================
# EOM-IP-CCSD and EOM-EA-CCSD
# ============================================================
# IP = Ionization Potential (remove electron → cation)
# EA = Electron Affinity (add electron → anion)

# EOM-IP-CCSD
myip = mycc.EOMIPSingles()
myip.nroots = 3
myip.verbose = 0
E_ip, _ = myip.kernel()
print("EOM-IP-CCSD Ionization Potentials:")
for i, E in enumerate(E_ip):
    print(f"  IP {i+1}: {E*27.211:.4f} eV")

# Koopmans' theorem comparison (HF orbitals)
n_occ = mol_small.nelectron // 2
koopmans = -mf_cc.mo_energy[n_occ-1] * 27.211
print(f"\nKoopmans' theorem (HOMO): {koopmans:.4f} eV")
print(f"(Negative of HOMO energy)")

---
<a id='chapter10'></a>
# Chapter 10: Multireference Methods (CASSCF/CASPT2/NEVPT2) 🌀

In [ ]:
# ============================================================
# When to use Multireference Methods?
# ============================================================
print("When Single-Reference Methods FAIL:")
print()
print("  1. Bond breaking / dissociation curves")
print("  2. Diradicals and biradicals")
print("  3. Transition metal complexes with near-degenerate d-levels")
print("  4. Excited states with doubly-excited character")
print("  5. Aromatic transition states (e.g., Bergman cyclization)")
print("  6. T1 diagnostic > 0.02 (single reference) or > 0.05 (double reference)")
print()
print("Multireference Method Ladder:")
print("  CASSCF                → qualitatively correct MR wavefunction")
print("  CASSCF + NEVPT2/CASPT2 → adds dynamic correlation")
print("  MRCISD                → more accurate but expensive")
print("  FCIQMC / DMRG         → for larger active spaces")

In [ ]:
# ============================================================
# CASSCF — Complete Active Space SCF
# ============================================================
from pyscf import mcscf

# O2 molecule — famous diradical
mol_o2 = gto.M(
    atom = 'O 0 0 0; O 0 0 1.21',
    basis = 'cc-pVDZ',
    spin = 2,        # triplet ground state
    charge = 0,
    symmetry = True,
    verbose = 0
)

# First do RHF/UHF
mf_o2 = scf.UHF(mol_o2)
mf_o2.verbose = 0
mf_o2.kernel()

# CASSCF(12,8) — 12 electrons in 8 orbitals
# Active space: 2σ, 2σ*, 1π×2, 1π*×2 = 8 orbitals
cas = mcscf.CASSCF(mf_o2, 8, 12)  # (norb, nelec)
cas.verbose = 4
cas.max_cycle_macro = 100
cas.conv_tol = 1e-8
E_casscf = cas.kernel()[0]

print(f"\nO₂ CASSCF(12,8)/cc-pVDZ: {E_casscf:.10f} Eh")
print(f"Converged: {cas.converged}")

In [ ]:
# ============================================================
# Selecting the Active Space — Critical Step!
# ============================================================
print("Active Space Selection Guidelines:")
print()
print("Rule of thumb: include all bonds being broken/formed")
print("and all near-degenerate orbitals.")
print()

# Natural orbital analysis helps identify active space
# Orbitals with occupation 0.02–1.98 are 'active'

# From UHF natural orbitals (good starting point)
dm1_uhf = mf_o2.make_rdm1()
dm1_tot = dm1_uhf[0] + dm1_uhf[1]  # total density matrix
occ_nums, nat_orbs = np.linalg.eigh(dm1_tot)
occ_nums = occ_nums[::-1]

print("UHF Natural Orbital Occupation Numbers:")
print("(Fractional occupations 0.02-1.98 → include in active space)")
print()
for i, n in enumerate(occ_nums):
    active = "← ACTIVE" if 0.02 < n < 1.98 else ""
    print(f"  NO {i:2d}: {n:.4f} {active}")

In [ ]:
# ============================================================
# NEVPT2 — N-Electron Valence State PT2
# ============================================================
# Adds dynamic correlation on top of CASSCF — more numerically stable than CASPT2
from pyscf import mrpt

# Smaller example: N2 with (10e, 8o) active space
mol_n2 = gto.M(
    atom = 'N 0 0 0; N 0 0 1.1',
    basis = 'cc-pVDZ',
    symmetry = True,
    verbose = 0
)
mf_n2 = scf.RHF(mol_n2)
mf_n2.verbose = 0
mf_n2.kernel()

# CASSCF(10,8)
cas_n2 = mcscf.CASSCF(mf_n2, 8, 10)
cas_n2.verbose = 0
E_cas = cas_n2.kernel()[0]

# NEVPT2 on top of CASSCF
nevpt2 = mrpt.NEVPT2(cas_n2)
nevpt2.verbose = 3
E_nevpt2 = nevpt2.kernel()

print(f"\nN₂/cc-pVDZ:")
print(f"  HF:           {mf_n2.e_tot:.8f} Eh")
print(f"  CASSCF(10,8): {E_cas:.8f} Eh")
print(f"  NEVPT2:       {E_nevpt2:.8f} Eh")

In [ ]:
# ============================================================
# CASCI State-Averaging (Multiple Roots)
# ============================================================
# State-averaged CASSCF: optimize orbitals for multiple states at once
# Essential for: excited state PES, conical intersections

mol_form = gto.M(
    atom = '''
        C   0.000  0.000  0.000
        O   0.000  0.000  1.220
        H   0.000  0.945 -0.540
        H   0.000 -0.945 -0.540
    ''',
    basis = 'cc-pVDZ',
    verbose = 0
)
mf_form = scf.RHF(mol_form)
mf_form.verbose = 0
mf_form.kernel()

# State-averaged CASSCF over 3 singlet states
# weights: [w_S0, w_S1, w_S2] — equal weights
n_states = 3
weights = np.ones(n_states) / n_states

sa_cas = mcscf.CASSCF(mf_form, 6, 6)  # (6e, 6o): CO π system
sa_cas = sa_cas.state_average_(weights)
sa_cas.verbose = 3
sa_cas.kernel()

# Extract individual state energies
print("\nState-Averaged CASSCF energies:")
for i, E in enumerate(sa_cas.e_states):
    print(f"  S{i}: {E:.8f} Eh  ({(E-sa_cas.e_states[0])*27.211:.4f} eV above S0)")

---
<a id='chapter11'></a>
# Chapter 11: Relativistic Methods ⚡

For heavy elements (Z > 36), relativistic effects are crucial for accuracy.

In [ ]:
# ============================================================
# Scalar Relativistic: ZORA and X2C
# ============================================================
print("Relativistic Methods in PySCF:")
print()
print("1. ECPs (Effective Core Potentials) — implicit relativistic")
print("   Uses def2-SVP, def2-TZVP etc. which include ECPs for heavy atoms")
print("   Simple, fast — recommended for most transition metals")
print()
print("2. X2C (eXact 2-Component) — scalar relativistic")
print("   Explicitly includes scalar relativistic effects")
print("   Required for 4d/5d metals, lanthanides, actinides")
print()
print("3. 4-Component Dirac (DHF/DKS) — full relativistic")
print("   Includes spin-orbit coupling explicitly")
print("   Required for heavy-atom spin-orbit effects")

# Example: Au atom (gold — famously relativistic)
mol_au = gto.M(
    atom = 'Au 0 0 0',
    basis = 'def2-SVP',    # ECP-based basis
    charge = 0,
    spin = 1,              # Au: [Xe]4f14 5d10 6s1 → doublet
    verbose = 0
)
print(f"\nAu atom: {mol_au.natm} atom, {mol_au.nao_nr()} AOs")
print(f"ECP included: {mol_au.has_ecp()}")

In [ ]:
# ============================================================
# X2C (eXact 2-Component) Hamiltonian
# ============================================================
from pyscf import x2c

# Water with X2C (to illustrate — light atoms won't show much difference)
mol_h2o = gto.M(
    atom = 'O 0 0 0.117; H 0 0.757 -0.469; H 0 -0.757 -0.469',
    basis = 'cc-pVDZ',
    verbose = 0
)

# Non-relativistic RHF
mf_nr = scf.RHF(mol_h2o)
mf_nr.verbose = 0
E_nr = mf_nr.kernel()

# X2C scalar relativistic RHF
mf_x2c = scf.RHF(mol_h2o).x2c()
mf_x2c.verbose = 0
E_x2c = mf_x2c.kernel()

print(f"H₂O/cc-pVDZ:")
print(f"  Non-relativistic: {E_nr:.10f} Eh")
print(f"  X2C:              {E_x2c:.10f} Eh")
print(f"  Difference:       {(E_x2c-E_nr)*1000:.6f} mEh  (tiny for light atoms)")

print("\nFor heavy atoms like Au, Pb, Tl: use X2C + ANO-RCC or DKH basis sets")

In [ ]:
# ============================================================
# 4-Component Dirac-Hartree-Fock
# ============================================================
from pyscf import gto, scf

# Hydrogen-like atom for illustration
mol_dhf = gto.M(
    atom = 'H 0 0 0',
    basis = 'sto-3g',
    verbose = 0
)

# 4-component Dirac-HF
mf_dhf = scf.DHF(mol_dhf)
mf_dhf.verbose = 3
E_dhf = mf_dhf.kernel()
print(f"\nDHF energy for H: {E_dhf:.8f} Eh")

# For production, use: mol.basis = 'dyall.vdz' (Dyall basis sets)
# and mol.spin to set spinors correctly

---
<a id='chapter12'></a>
# Chapter 12: Periodic Systems (Solid State / PBC) 🔷

In [ ]:
# ============================================================
# Periodic Boundary Conditions (PBC) in PySCF
# ============================================================
from pyscf.pbc import gto as pbcgto
from pyscf.pbc import scf as pbcscf
from pyscf.pbc import df as pbcdf
import numpy as np

# Face-centered cubic (FCC) Diamond structure
a = 3.567  # Angstrom (diamond lattice constant)

cell = pbcgto.Cell()
cell.atom = '''
    C   0.000  0.000  0.000
    C   0.890  0.890  0.890
'''
cell.a = [[a/2, a/2, 0],
          [a/2, 0,   a/2],
          [0,   a/2, a/2]]   # FCC lattice vectors (Angstrom)

cell.basis = 'gth-szv'       # GTH pseudopotential basis (for periodic)
cell.pseudo = 'gth-pade'     # GTH-PADE pseudopotential
cell.verbose = 4
cell.max_memory = 2000       # MB
cell.build()

print(f"Diamond unit cell:")
print(f"  Atoms: {cell.natm}")
print(f"  Basis functions: {cell.nao_nr()}")
print(f"  Volume: {cell.vol:.4f} Å³")

In [ ]:
# ============================================================
# Gamma-Point DFT Calculation
# ============================================================
# Gamma point: k=(0,0,0) — simplest calculation

mf_pbc = pbcscf.RKS(cell)
mf_pbc.xc = 'pbe'
mf_pbc.with_df = pbcdf.FFTDF(cell)  # Density fitting for PBC
mf_pbc.verbose = 3
E_pbc = mf_pbc.kernel()

print(f"\nDiamond PBE/gth-szv @ Γ-point: {E_pbc:.8f} Eh/cell")
print(f"Per atom: {E_pbc/2:.8f} Eh")

In [ ]:
# ============================================================
# k-Point Sampling — Band Structure
# ============================================================
from pyscf.pbc import scf as pbcscf

# k-point sampling with Monkhorst-Pack grid
kpts = cell.make_kpts([2, 2, 2])   # 2×2×2 k-mesh
print(f"k-points: {len(kpts)}")
print(f"k-grid: {kpts.shape}")

# KRHF/KRKS for k-point-sampled calculation
kmf = pbcscf.KRKS(cell, kpts)
kmf.xc = 'pbe'
kmf.verbose = 3
E_k = kmf.kernel()

print(f"\nDiamond PBE/gth-szv (2×2×2 k-mesh): {E_k:.8f} Eh/cell")

# Get band gap
n_occ_per_k = cell.nelectron // 2
print(f"\nBand gap estimate from k-mesh:")
homo_k = max([kmf.mo_energy[k][n_occ_per_k-1] for k in range(len(kpts))])
lumo_k = min([kmf.mo_energy[k][n_occ_per_k]   for k in range(len(kpts))])
gap = (lumo_k - homo_k) * 27.211
print(f"  HOMO (VBM): {homo_k*27.211:.4f} eV")
print(f"  LUMO (CBM): {lumo_k*27.211:.4f} eV")
print(f"  Gap:        {gap:.4f} eV  (exp diamond: ~5.5 eV)")

In [ ]:
# ============================================================
# Band Structure Calculation
# ============================================================
from pyscf.pbc.tools.pbc import get_monkhorst_pack_size

# High-symmetry path for FCC lattice: Γ → X → W → K → Γ → L
# For diamond (FCC): standard path in units of reciprocal lattice
Gamma = [0.0,  0.0,  0.0]
X     = [0.5,  0.0,  0.5]
L     = [0.5,  0.5,  0.5]
W     = [0.5,  0.25, 0.75]

n_kpts_per_seg = 10
# Build k-path
def kpath(k1, k2, n):
    return [np.array(k1) + (np.array(k2)-np.array(k1)) * i/(n-1) for i in range(n)]

kpath_points = (kpath(Gamma, X, n_kpts_per_seg) + 
                kpath(X, L, n_kpts_per_seg)[1:] + 
                kpath(L, Gamma, n_kpts_per_seg)[1:])
kpts_band = cell.get_abs_kpts(kpath_points)

print(f"Band structure k-path: {len(kpts_band)} k-points")

# Use scf result from Γ calculation as starting point
# This requires the converged KRKS calc above
from pyscf.pbc.dft import KRKS
band_mf = kmf.get_bands(kpts_band)
bands_eV = np.array(band_mf[0]) * 27.211  # shape: (nk, nbands)

# Plot band structure
plt.figure(figsize=(8, 6))
E_fermi = homo_k * 27.211
for ib in range(bands_eV.shape[1]):
    plt.plot(range(len(kpts_band)), bands_eV[:, ib] - E_fermi, 'b-', lw=1, alpha=0.8)

plt.axhline(y=0, color='k', linestyle='--', lw=0.5, alpha=0.5, label='VBM')
plt.axvline(x=n_kpts_per_seg-1,   color='gray', lw=0.5, alpha=0.5)
plt.axvline(x=2*n_kpts_per_seg-2, color='gray', lw=0.5, alpha=0.5)
plt.xticks([0, n_kpts_per_seg-1, 2*n_kpts_per_seg-2, len(kpts_band)-1],
           ['Γ', 'X', 'L', 'Γ'])
plt.ylabel('Energy − E_VBM (eV)')
plt.ylim(-5, 10)
plt.title('Diamond Band Structure (PBE/gth-szv)')
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

---
<a id='chapter13'></a>
# Chapter 13: Solvation & Embedding 🌊

In [ ]:
# ============================================================
# PCM — Polarizable Continuum Model
# ============================================================
from pyscf import solvent

mol = gto.M(
    atom = 'O 0 0 0.117; H 0 0.757 -0.469; H 0 -0.757 -0.469',
    basis = 'cc-pVDZ',
    verbose = 0
)

# Gas phase
mf_gas = dft.RKS(mol)
mf_gas.xc = 'b3lyp'
mf_gas.verbose = 0
E_gas = mf_gas.kernel()

# With PCM solvation (water, ε=78.4)
mf_pcm = dft.RKS(mol).PCM()
mf_pcm.xc = 'b3lyp'
mf_pcm.with_solvent.eps = 78.4      # dielectric constant
mf_pcm.with_solvent.method = 'iefpcm'  # IEF-PCM (default)
mf_pcm.verbose = 3
E_pcm = mf_pcm.kernel()

delta_G_solv = (E_pcm - E_gas) * 2625.5  # kJ/mol
print(f"\nSolvation Free Energy (PCM):")
print(f"  Gas phase:   {E_gas:.8f} Eh")
print(f"  In water:    {E_pcm:.8f} Eh")
print(f"  ΔG_solv:     {delta_G_solv:.2f} kJ/mol")

In [ ]:
# ============================================================
# PCM Methods Available
# ============================================================
print("PCM Variants in PySCF:")
pcm_methods = [
    ('C-PCM',   'CPCM',   'Conductor-like PCM. Fastest.'),
    ('IEF-PCM', 'IEFPCM', 'Integral Equation Formalism PCM. Default.'),
    ('SS(V)PE', 'SSVPE',  'Surface and Simulation of Volume Polarization.'),
    ('COSMO',   'COSMO',  'COnductor-like Screening MOdel.'),
]
for name, key, desc in pcm_methods:
    print(f"  {name:10s} (method='{key}'): {desc}")

print()
print("Common solvents (ε):")
solvents = [
    ('Water',          78.4),
    ('DMSO',           46.7),
    ('Acetonitrile',   37.5),
    ('Methanol',       32.6),
    ('Ethanol',        24.3),
    ('Acetone',        20.7),
    ('THF',            7.58),
    ('Chloroform',     4.81),
    ('Toluene',        2.37),
    ('Hexane',         1.88),
    ('Vacuum',         1.00),
]
for name, eps in solvents:
    print(f"  {name:20s}: ε = {eps}")

In [ ]:
# ============================================================
# QM/MM — Quantum Mechanics / Molecular Mechanics
# ============================================================
from pyscf import qmmm

# QM region: water molecule
mol_qm = gto.M(
    atom = 'O 0 0 0.117; H 0 0.757 -0.469; H 0 -0.757 -0.469',
    basis = 'cc-pVDZ',
    verbose = 0
)

# MM point charges (surrounding environment)
# Format: [charge, (x, y, z)]  — Angstrom coordinates
mm_charges = np.array([-0.834, 0.417, 0.417,  -0.834])  # water charges
mm_coords  = np.array([
    [ 0.000,  0.000,  3.000],  # O of nearby water
    [ 0.000,  0.757,  3.586],  # H
    [ 0.000, -0.757,  3.586],  # H
    [ 0.000,  0.000, -3.000],  # Another O
]) * 1.8897  # Angstrom → Bohr

# QM/MM RHF
mf_qmmm = qmmm.mm_charge(scf.RHF(mol_qm), mm_coords, mm_charges)
mf_qmmm.verbose = 3
E_qmmm = mf_qmmm.kernel()

# Gas phase for comparison
mf_vac = scf.RHF(mol_qm)
mf_vac.verbose = 0
E_vac = mf_vac.kernel()

print(f"\nQM/MM Results:")
print(f"  Gas phase:   {E_vac:.8f} Eh")
print(f"  QM/MM:       {E_qmmm:.8f} Eh")
print(f"  MM effect:   {(E_qmmm-E_vac)*2625.5:.4f} kJ/mol")

---
<a id='chapter14'></a>
# Chapter 14: Wavefunction Analysis 🔍

In [ ]:
# ============================================================
# Localized Orbitals
# ============================================================
# Canonical MOs are delocalized. Localized orbitals give chemical insight.
from pyscf import lo

mol = gto.M(
    atom = '''
        C   0.000  0.000  0.000
        H   0.630  0.630  0.630
        H  -0.630 -0.630  0.630
        H  -0.630  0.630 -0.630
        H   0.630 -0.630 -0.630
    ''',
    basis = 'cc-pVDZ',
    verbose = 0
)
mf = scf.RHF(mol)
mf.verbose = 0
mf.kernel()

n_occ = mol.nelectron // 2
mo_occ = mf.mo_coeff[:, :n_occ]  # Occupied MOs only

# Foster-Boys Localization
boys_loc = lo.Boys(mol, mo_occ)
mo_boys = boys_loc.kernel()

# Pipek-Mezey Localization
pm_loc = lo.PM(mol, mo_occ)
mo_pm = pm_loc.kernel()

# Intrinsic Atomic Orbitals (IAO)
mo_iao = lo.iao.iao(mol, mo_occ)

print("Localized Orbitals (CH₄):")
print(f"  Boys:  {mo_boys.shape[1]} localized orbitals")
print(f"  PM:    {mo_pm.shape[1]} localized orbitals")
print(f"  IAO:   {mo_iao.shape[1]} orbitals")

# Write to cube for visualization
from pyscf.tools import cubegen
cubegen.orbital(mol, '/tmp/ch4_boys_1.cube', mo_boys[:, 0])
print("\nWritten Boys LO #1 to /tmp/ch4_boys_1.cube")

In [ ]:
# ============================================================
# Electron Density Analysis
# ============================================================
# Evaluate density at grid points

from pyscf import dft

mol = gto.M(
    atom = 'O 0 0 0.117; H 0 0.757 -0.469; H 0 -0.757 -0.469',
    basis = 'cc-pVDZ',
    verbose = 0
)
mf = scf.RHF(mol)
mf.verbose = 0
mf.kernel()
dm = mf.make_rdm1()

# Evaluate on a grid
x = np.linspace(-3, 3, 50)   # Bohr
y = np.zeros(50)
z = np.zeros(50)
coords = np.column_stack([x, y, z])

# Compute AO values at grid points
ao_vals = mol.eval_gto('GTOval', coords)  # shape: (n_grid, n_ao)

# Electron density: ρ(r) = Σ_μν P_μν φ_μ(r) φ_ν(r)
rho = np.einsum('gi,ij,gj->g', ao_vals, dm, ao_vals)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(x * 0.529177, rho, 'b-', lw=2)  # convert Bohr to Å
plt.xlabel('x (Å)')
plt.ylabel('ρ(x, 0, 0)')
plt.title('Electron Density along x-axis (H₂O)')
plt.grid(True, alpha=0.3)

# Fukui function: f⁺(r) = ρ_N+1 - ρ_N (nucleophilic attack)
# Approximate: frontier orbital contribution
homo_ao = ao_vals @ mf.mo_coeff[:, mol.nelectron//2 - 1]
lumo_ao = ao_vals @ mf.mo_coeff[:, mol.nelectron//2]

plt.subplot(1, 2, 2)
plt.plot(x * 0.529177, homo_ao**2, 'r-', lw=2, label='HOMO²')
plt.plot(x * 0.529177, lumo_ao**2, 'b--', lw=2, label='LUMO²')
plt.xlabel('x (Å)')
plt.ylabel('|φ(x)|²')
plt.title('Frontier Orbital Densities')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Natural Bond Orbital (NBO) Analysis
# ============================================================
# NBO gives chemical bonding picture: bonds, lone pairs, hyperconjugation
# Requires: pip install nbo7  or use pyscf-nbo

print("NBO Analysis:")
print()
print("Option 1: External NBO7/NBO6 program")
print("  from pyscf.tools import nbo")
print("  nbo.make_nao_fcidump(mf, 'nbo_input.fcidump')")
print("  # Run NBO7 externally")
print()
print("Option 2: Built-in NAO (Natural Atomic Orbitals)")

# Natural Atomic Orbitals (built-in)
from pyscf import lo
mol_h2o = gto.M(
    atom = 'O 0 0 0.117; H 0 0.757 -0.469; H 0 -0.757 -0.469',
    basis = 'cc-pVDZ', verbose = 0
)
mf_h2o = scf.RHF(mol_h2o)
mf_h2o.verbose = 0
mf_h2o.kernel()

# Natural Population Analysis
mf_h2o.analyze(verbose=4)

---
<a id='chapter15'></a>
# Chapter 15: Advanced Topics & Real Workflows 🏆

In [ ]:
# ============================================================
# 15.1 Potential Energy Surface Scan
# ============================================================

def pes_scan_distance(atom1, atom2, r_min, r_max, n_points, 
                       fixed_atoms_str, basis, method='B3LYP'):
    """
    1D PES scan along a bond distance.
    """
    distances = np.linspace(r_min, r_max, n_points)
    energies  = []
    
    for r in distances:
        # Build molecule with current bond length
        atom_str = fixed_atoms_str.format(r=r)
        mol = gto.M(atom=atom_str, basis=basis, verbose=0)
        
        mf = dft.RKS(mol) if method != 'HF' else scf.RHF(mol)
        if hasattr(mf, 'xc'):
            mf.xc = method.lower()
        mf.verbose = 0
        E = mf.kernel()
        energies.append(E)
    
    return distances, np.array(energies)

# Scan H2O OH bond length
atom_template = '''
    O   0.000  0.000  0.000
    H   0.000  0.000  {r:.4f}
    H   0.000  {r2:.4f} {z2:.4f}
'''

# Simpler: just diatomic scan
print("Scanning H₂ PES...")
r_vals  = np.linspace(0.5, 4.0, 25)  # Angstrom
E_rhf   = []
E_b3lyp = []

for r in r_vals:
    mol_h2 = gto.M(
        atom = f'H 0 0 0; H 0 0 {r}',
        basis = 'cc-pVDZ',
        verbose = 0
    )
    # RHF
    mf = scf.RHF(mol_h2); mf.verbose = 0
    E_rhf.append(mf.kernel())
    
    # B3LYP
    mf_dft = dft.RKS(mol_h2); mf_dft.xc='b3lyp'; mf_dft.verbose=0
    E_b3lyp.append(mf_dft.kernel())

E_rhf   = np.array(E_rhf)
E_b3lyp = np.array(E_b3lyp)

plt.figure(figsize=(10, 5))
plt.plot(r_vals, (E_rhf   - E_rhf.min())   * 2625.5, 'b-o', ms=4, lw=2, label='RHF')
plt.plot(r_vals, (E_b3lyp - E_b3lyp.min()) * 2625.5, 'r-s', ms=4, lw=2, label='B3LYP')
plt.xlabel('H–H Distance (Å)')
plt.ylabel('Relative Energy (kJ/mol)')
plt.title('H₂ Potential Energy Surface')
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(-10, 300)
plt.axvline(x=0.741, color='k', linestyle='--', alpha=0.5, label='r_eq')
plt.tight_layout()
plt.show()

# Note: RHF fails to dissociate correctly (unphysical at large r)
# B3LYP also has issues — UHF/UDFT needed for proper dissociation

In [ ]:
# ============================================================
# 15.2 Reaction Energy Calculation
# ============================================================
# Example: H2O → H· + OH·  (homolytic O-H bond dissociation)

def single_point(atom_str, basis, xc, charge=0, spin=0):
    mol = gto.M(atom=atom_str, basis=basis, charge=charge, spin=spin, verbose=0)
    if xc.upper() == 'HF':
        if spin > 0:
            mf = scf.UHF(mol)
        else:
            mf = scf.RHF(mol)
    else:
        mf = dft.UKS(mol) if spin > 0 else dft.RKS(mol)
        mf.xc = xc
    mf.verbose = 0
    return mf.kernel()

basis = 'aug-cc-pVTZ'
xc    = 'b3lyp'

E_h2o = single_point('O 0 0 0.117; H 0 0.757 -0.469; H 0 -0.757 -0.469', basis, xc, spin=0)
E_oh  = single_point('O 0 0 0; H 0 0 0.97', basis, xc, spin=1)
E_h   = single_point('H 0 0 0', basis, xc, spin=1)

# Bond dissociation energy
BDE = (E_oh + E_h - E_h2o) * 2625.5  # kJ/mol

print("Homolytic O-H Bond Dissociation of H₂O:")
print(f"  H₂O:   {E_h2o:.8f} Eh")
print(f"  OH·:   {E_oh:.8f} Eh")
print(f"  H·:    {E_h:.8f} Eh")
print(f"  BDE:   {BDE:.2f} kJ/mol")
print(f"  BDE:   {BDE/4.184:.2f} kcal/mol")
print(f"  Experimental: 498 kJ/mol (119 kcal/mol)")

In [ ]:
# ============================================================
# 15.3 Composite Methods (G4-like thermochemistry)
# ============================================================
# Composite methods combine multiple calc levels for near-CBS accuracy

def g4_like_energy(mol_atom_str, charge=0, spin=0):
    """
    Simplified G4-like composite energy.
    E_composite ≈ CCSD(T)/small + MP2/large - MP2/small + ZPE(scaled)
    """
    # CCSD(T) with small basis
    mol_s = gto.M(atom=mol_atom_str, basis='cc-pVDZ', charge=charge, spin=spin, verbose=0)
    mf_s  = scf.RHF(mol_s) if spin==0 else scf.UHF(mol_s)
    mf_s.verbose = 0; mf_s.kernel()
    
    mp2_s = mp.MP2(mf_s); mp2_s.verbose = 0
    e_mp2_s, _ = mp2_s.kernel()
    
    ccsd_s = cc.CCSD(mf_s); ccsd_s.verbose = 0; ccsd_s.kernel()
    e_t_s  = ccsd_s.ccsd_t()
    
    E_ccsd_t = mf_s.e_tot + ccsd_s.e_corr + e_t_s
    
    # MP2 with large basis
    mol_l = gto.M(atom=mol_atom_str, basis='cc-pVTZ', charge=charge, spin=spin, verbose=0)
    mf_l  = scf.RHF(mol_l) if spin==0 else scf.UHF(mol_l)
    mf_l.verbose = 0; mf_l.kernel()
    
    mp2_l = mp.MP2(mf_l); mp2_l.verbose = 0
    e_mp2_l, _ = mp2_l.kernel()
    
    # Composite
    E_composite = E_ccsd_t + (e_mp2_l - e_mp2_s)
    
    return E_composite, E_ccsd_t, e_mp2_s, e_mp2_l

print("Composite energy for H₂O:")
E_comp, E_high, E_low_s, E_low_l = g4_like_energy(
    'O 0 0 0.117; H 0 0.757 -0.469; H 0 -0.757 -0.469'
)
print(f"  CCSD(T)/cc-pVDZ:     {E_high:.8f} Eh")
print(f"  MP2/cc-pVDZ corr:    {E_low_s:.8f} Eh")
print(f"  MP2/cc-pVTZ corr:    {E_low_l:.8f} Eh")
print(f"  Composite energy:    {E_comp:.8f} Eh")

In [ ]:
# ============================================================
# 15.4 GPU Acceleration (gpu4pyscf)
# ============================================================
print("GPU Acceleration with gpu4pyscf:")
print()
print("Install: pip install gpu4pyscf")
print("Requires: NVIDIA GPU with CUDA")
print()

try:
    from gpu4pyscf import dft as gpu_dft
    print("gpu4pyscf available!")
    
    mol = gto.M(atom='O 0 0 0.117; H 0 0.757 -0.469; H 0 -0.757 -0.469',
                basis='def2-TZVP', verbose=0)
    
    # GPU DFT — identical API to CPU version!
    mf_gpu = gpu_dft.RKS(mol)
    mf_gpu.xc = 'b3lyp'
    E_gpu = mf_gpu.kernel()
    print(f"GPU B3LYP energy: {E_gpu:.8f} Eh")
    
except ImportError:
    print("gpu4pyscf not installed. Example:")
    print('''
  from gpu4pyscf import dft as gpu_dft
  # Same API as CPU — just import from gpu4pyscf instead!
  mf = gpu_dft.RKS(mol).to_gpu()  # or explicitly move to GPU
  mf.xc = 'b3lyp'
  E = mf.kernel()
  
  # Typical GPU speedups:
  # - DFT:     10-50x faster
  # - DF-MP2:  5-20x faster
  # - Hessian: 10-30x faster
  ''')

In [ ]:
# ============================================================
# 15.5 Comprehensive Method Comparison
# ============================================================

# Atomization energy of H2O: H2O → 2H + O
# Experimental: 971.6 kJ/mol

basis = 'aug-cc-pVTZ'

# Define calculations
species = {
    'H2O': ('O 0 0 0.117; H 0 0.757 -0.469; H 0 -0.757 -0.469', 0, 0),
    'O':   ('O 0 0 0',   0, 2),
    'H':   ('H 0 0 0',   0, 1),
}

methods = ['HF', 'BLYP', 'B3LYP', 'PBE0', 'CAM-B3LYP', 'MP2']
results = {}

for method in methods:
    energies = {}
    for name, (atom_str, chg, spin) in species.items():
        mol = gto.M(atom=atom_str, basis=basis, charge=chg, spin=spin, verbose=0)
        if method == 'HF':
            mf = scf.UHF(mol) if spin > 0 else scf.RHF(mol)
        elif method == 'MP2':
            mf = scf.UHF(mol) if spin > 0 else scf.RHF(mol)
            mf.verbose = 0; mf.kernel()
            mp2_obj = mp.MP2(mf); mp2_obj.verbose = 0
            e_corr, _ = mp2_obj.kernel()
            energies[name] = mf.e_tot + e_corr
            continue
        else:
            mf = dft.UKS(mol) if spin > 0 else dft.RKS(mol)
            mf.xc = method.lower()
            mf.grids.level = 4
        mf.verbose = 0
        energies[name] = mf.kernel()
    
    AE = (energies['O'] + 2*energies['H'] - energies['H2O']) * 2625.5
    results[method] = {'AE (kJ/mol)': round(AE, 2), 'Error (kJ/mol)': round(AE - 971.6, 2)}

df_comp = pd.DataFrame(results).T
print("Atomization Energy of H₂O (aug-cc-pVTZ):")
print(df_comp.to_string())
print(f"\nExperimental (atomization): 971.6 kJ/mol")

In [ ]:
# ============================================================
# 15.6 PySCF Ecosystem
# ============================================================
print("The PySCF Ecosystem:")
print()

ecosystem = [
    ('gpu4pyscf',        'pip', 'GPU acceleration for DFT, MP2, Hessian'),
    ('pyscf-forge',      'pip', 'Community extensions (extra methods)'),
    ('pyscf-geomopt',    'pip', 'Geometry optimization (geometric/berny)'),
    ('pyberny',          'pip', 'Geometry optimizer by Jiri Hermann'),
    ('geometric',        'pip', 'High-quality geometry optimizer'),
    ('dispersion',       'pip', 'D3/D4 dispersion corrections'),
    ('dftd3',            'pip', 'DFT-D3 dispersion'),
    ('Block2',           'pip', 'DMRG for large active spaces (by Garnet Chan)'),
    ('Dice',             'src', 'Selected CI (SHCI) for large active spaces'),
    ('OpenFermion',      'pip', 'Quantum computing integration'),
    ('QMCPy',            'pip', 'Quantum Monte Carlo'),
    ('socutils',         'pip', 'Spin-orbit coupling'),
    ('pyscf-ipu',        'pip', 'JAX/IPU-accelerated PySCF'),
]

print(f"{'Package':20s} {'Install':8s} {'Purpose'}")
print("-" * 75)
for pkg, ins, purpose in ecosystem:
    print(f"  {pkg:18s} {ins:8s} {purpose}")

print()
print("Machine Learning interoperability:")
print("  - DeePMD-kit: DPMD potentials trained on PySCF data")
print("  - SchNetPack:  ML potentials with PySCF training data")
print("  - MACE:       Equivariant ML FFs trained on QM data")
print("  - TensorMol:  DNN potentials")

In [ ]:
# ============================================================
# 15.7 Parallel Computing
# ============================================================
import os

print("Parallel Computing Options in PySCF:")
print()

print("1. OpenMP (shared memory) — default")
print(f"   Current threads: {lib.num_threads()}")
print(f"   CPU count:       {os.cpu_count()}")
print("   Set with: lib.num_threads(N)")
print("   Or env:   export OMP_NUM_THREADS=8")
print()

print("2. MPI (distributed memory) via mpi4py")
print("   pip install mpi4py")
print("   mpirun -np 8 python my_script.py")
print()

print("3. Embarrassingly parallel: scan, benchmark")
from concurrent.futures import ProcessPoolExecutor

def calc_energy(r):
    """Worker function for parallel PES scan."""
    mol = gto.M(atom=f'H 0 0 0; H 0 0 {r}', basis='sto-3g', verbose=0)
    mf = scf.RHF(mol); mf.verbose = 0
    return mf.kernel()

r_points = np.linspace(0.5, 3.0, 10)
with ProcessPoolExecutor(max_workers=4) as executor:
    energies_parallel = list(executor.map(calc_energy, r_points))

print(f"\nParallel PES scan (10 points, 4 workers):")
for r, E in zip(r_points, energies_parallel):
    print(f"  r={r:.3f} Å: E={E:.6f} Eh")

---

# 🎓 Summary & Key Takeaways

Congratulations — you've mastered PySCF from zero to hero! Here's what you've covered:

| Topic | Key Objects / Functions |
|-------|------------------------|
| Molecule setup | `gto.M()`, `gto.Mole()`, `mol.build()` |
| Basis sets | STO-3G → cc-pVTZ, def2-TZVP, aug-cc-pVDZ |
| HF | `scf.RHF()`, `scf.UHF()`, `scf.ROHF()` |
| DFT | `dft.RKS()`, `dft.UKS()`, `.xc = 'b3lyp'` |
| Molecular properties | Dipole, polarizability, NMR, ESP charges |
| MP2 | `mp.MP2()`, `mp.DFMP2()` |
| CCSD(T) | `cc.CCSD()`, `.ccsd_t()` |
| FCI / CISD | `fci.FCI()`, `ci.CISD()` |
| Geometry opt | `geomopt.optimize()`, `.nuc_grad_method()` |
| Frequencies | `mf.Hessian()`, `harmonic_analysis()` |
| TDDFT | `mf.TDDFT()`, `td.oscillator_strength()` |
| EOM-CCSD | `mycc.EOMEESingles()`, `EOMIPSingles()` |
| CASSCF | `mcscf.CASSCF(mf, norb, nelec)` |
| NEVPT2 | `mrpt.NEVPT2(cas)` |
| Relativistic | `.x2c()`, `scf.DHF()` |
| Periodic (PBC) | `pbcgto.Cell()`, `pbcscf.KRKS()` |
| Solvation | `dft.RKS(mol).PCM()` |
| QM/MM | `qmmm.mm_charge()` |
| Localized MOs | `lo.Boys()`, `lo.PM()`, `lo.IAO()` |

## Method Selection Flowchart

```
System?  ─── small (<20 atoms)  ──→  CCSD(T)/CBS (gold standard)
         ─── medium (20-100)    ──→  DFT (B3LYP, PBE0, M06-2X)
         ─── large (>100)       ──→  DFT + RI + D3 dispersion
         ─── periodic           ──→  PBE / HSE06 + PBC
         ─── MR character?      ──→  CASSCF + NEVPT2
         ─── excited states?    ──→  TDDFT (large) / EOM-CCSD (small)
         ─── heavy atoms?       ──→  X2C / def2 + ECP
```

## Key Resources
- 📖 [PySCF Documentation](https://pyscf.org/)
- 📝 [PySCF User Guide](https://pyscf.org/user_guide.html)
- 🧪 [PySCF Examples](https://github.com/pyscf/pyscf/tree/master/examples)
- 📦 [PySCF Extensions](https://github.com/pyscf/pyscf-forge)
- 🎓 [Szabo & Ostlund — Modern Quantum Chemistry](https://store.doverpublications.com/0486691861.html)
- 🎓 [Jensen — Introduction to Computational Chemistry](https://www.wiley.com/en-us)
- 🔬 [Basis Set Exchange](https://www.basissetexchange.org/)

---
*Happy computing! ⚛️🔬*